In [1]:
import os
os.environ["PYTHONNET_RUNTIME"] = "coreclr"
os.environ["DOTNET_SYSTEM_DRAWING_USE_GDIPLUS"] = "1"

import clr
import numpy as np
import time
from datetime import timedelta
from pythonnet import load
from System import Environment, Array, String

# Load .NET Core runtime
load("coreclr")

# Try to import System.IO; fallback to os.chdir if it fails
try:
    from System.IO import Directory, Path, File
    use_dotnet_dir = True
except Exception as e:
    print(f"Note: System.IO import failed ({e}), using os.chdir instead")
    import os
    use_dotnet_dir = False

# Define DWSIM path
dwsimpath = "/usr/local/lib/dwsim/"

# Set working directory
if use_dotnet_dir:
    Directory.SetCurrentDirectory(dwsimpath)
else:
    os.chdir(dwsimpath)

# Add DWSIM assemblies
clr.AddReference(dwsimpath + "CapeOpen.dll")
clr.AddReference(dwsimpath + "DWSIM.Automation.dll")
clr.AddReference(dwsimpath + "DWSIM.Interfaces.dll")
clr.AddReference(dwsimpath + "DWSIM.GlobalSettings.dll")
clr.AddReference(dwsimpath + "DWSIM.SharedClasses.dll")
clr.AddReference(dwsimpath + "DWSIM.Thermodynamics.dll")
clr.AddReference(dwsimpath + "DWSIM.UnitOperations.dll")
clr.AddReference(dwsimpath + "DWSIM.Inspector.dll")
clr.AddReference(dwsimpath + "System.Buffers.dll")
clr.AddReference(dwsimpath + "DWSIM.Thermodynamics.ThermoC.dll")

# Now import DWSIM types
from DWSIM.Interfaces.Enums.GraphicObjects import ObjectType
from DWSIM.Thermodynamics import Streams, PropertyPackages
from DWSIM.UnitOperations import UnitOperations
from DWSIM.Automation import Automation3
from DWSIM.GlobalSettings import Settings

print("DWSIM imports successful!")

DWSIM imports successful!


In [2]:
# Create an instance of the Automation3 class from the DWSIM.Automation module
# This class provides methods for automating tasks in DWSIM, such as creating and manipulating flowsheets
interf = Automation3()

In [3]:
# Creates a flowsheet
sim = interf.CreateFlowsheet()

In [4]:
# Add Compounds
sim.AddCompound("Benzene")
sim.AddCompound("Toluene")

In [5]:
feed = sim.AddFlowsheetObject('Material Stream','feed')
distillate = sim.AddFlowsheetObject('Material Stream','distillate')
residue = sim.AddFlowsheetObject('Material Stream', 'residue')
c_duty = sim.AddFlowsheetObject('Energy Stream','C-Duty')
r_duty = sim.AddFlowsheetObject('Energy Stream','R-Duty')
sc = sim.AddFlowsheetObject('Shortcut Column','SC')

In [6]:
feed = feed.GetAsObject()
distillate = distillate.GetAsObject()
residue = residue.GetAsObject()
c_duty = c_duty.GetAsObject()
r_duty = r_duty.GetAsObject()
sc = sc.GetAsObject()

In [7]:
sim.ConnectObjects(feed.GraphicObject , sc.GraphicObject, -1, -1)
sim.ConnectObjects(sc.GraphicObject, distillate.GraphicObject, -1, -1)
sim.ConnectObjects(sc.GraphicObject, residue.GraphicObject, -1, -1)
sim.ConnectObjects(r_duty.GraphicObject, sc.GraphicObject, -1, -1)
sim.ConnectObjects(sc.GraphicObject, c_duty.GraphicObject, -1, -1)

In [8]:
sim.AutoLayout()

In [9]:
feed.SetOverallComposition(Array[float]([0.5, 0.5]))
feed.SetTemperature(300.0) # K
feed.SetPressure(101325.0) # Pa
feed.SetMassFlow(1.0) # kg/s

'feed: mass flow set to 1 kg/s'

In [10]:
# property package
Thermo_Package = sim.CreateAndAddPropertyPackage("Peng-Robinson (PR)")

In [11]:
# Calc Modes
sc.m_lightkey = "Benzene"
sc.m_heavykey = "Toluene"

sc.m_heavykeymolarfrac = 0.01
sc.m_lightkeymolarfrac = 0.01

sc.m_refluxratio = 1.5
sc.m_condenserpressure = 101325.0
sc.m_reboilerpressure = 101325.0

sc.CondenserType = 0 # Total condenser

In [12]:
# request a calculation
errors = interf.CalculateFlowsheet4(sim)
list(errors)

[]

In [13]:
# save file

fileNameToSave = Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.Desktop), "/workspace/00 FlowSheet Automation/11 Shortcut Column Automation/11 Shortcut Column.dwxmz")

interf.SaveFlowsheet(sim, fileNameToSave, True)